## Part A

In [1]:
# Workspace setup

from pathlib import Path
from datetime import datetime, timezone
import os, json, hashlib, requests

BASE_DIR = Path("/content/week3_project")
RAW_DIR = BASE_DIR / "data/raw"
PROCESSED_DIR = BASE_DIR / "data/processed"
PIPE_DIR = BASE_DIR / "data/pipeline_v1/json"
for p in [RAW_DIR, PROCESSED_DIR, PIPE_DIR]:
    p.mkdir(parents=True, exist_ok=True)
print("Workspace ready:", BASE_DIR)

Workspace ready: /content/week3_project


In [2]:
# OpenAlex API access

try:
    from google.colab import userdata
    API_KEY = userdata.get("OPENALEX_API_KEY")
except Exception:
    API_KEY = os.getenv("OPENALEX_API_KEY")
if not API_KEY:
    raise RuntimeError("Add OPENALEX_API_KEY to Colab Secrets.")
ENDPOINT = "https://api.openalex.org/works"
FIELDS = ",".join([
    "id","doi","title","publication_year","publication_date",
    "authorships","cited_by_count","open_access",
    "abstract_inverted_index","primary_location","type"
])
print("API key loaded securely.")

API key loaded securely.


In [3]:
# Ingest two JSON/API subsets

QUERIES = {"S01": "artificial intelligence", "S02": "machine learning"}
source_paths = {}

def fetch_query(query, per_page=10):
    params = {"search": query, "per_page": per_page,
              "select": FIELDS, "api_key": API_KEY}
    response = requests.get(ENDPOINT, params=params, timeout=30)
    response.raise_for_status()
    payload = response.json()
    if not isinstance(payload.get("results"), list):
        raise ValueError("OpenAlex response has no results list.")
    return payload, response.url

for sid, query in QUERIES.items():
    payload, request_url = fetch_query(query)
    record = {
        "source_id": sid, "provider": "OpenAlex",
        "endpoint": ENDPOINT, "query": query,
        "request_url": request_url,
        "retrieved_at_utc": datetime.now(timezone.utc).isoformat(),
        "api_meta": payload.get("meta", {}),
        "results": payload["results"]
    }
    path = RAW_DIR / f"{sid.lower()}_openalex.json"
    path.write_text(json.dumps(record, ensure_ascii=False, indent=2), encoding="utf-8")
    source_paths[sid] = path
    print(sid, query, "records:", len(payload["results"]), "saved:", path.name)
assert len(source_paths) == 2

S01 artificial intelligence records: 10 saved: s01_openalex.json
S02 machine learning records: 10 saved: s02_openalex.json


In [4]:
# Inspect the returned JSON and representative records

for sid, path in source_paths.items():
    data = json.loads(path.read_text(encoding="utf-8"))
    print("\\n", sid, "query:", data["query"])
    print("Retrieved:", data["retrieved_at_utc"])
    print("Records:", len(data["results"]))
    if data["results"]:
        sample = data["results"][0]
        print("Keys:", list(sample.keys()))
        print("Title:", sample.get("title"))
        print("Year:", sample.get("publication_year"))
        print("Abstract index present:", bool(sample.get("abstract_inverted_index")))

\n S01 query: artificial intelligence
Retrieved: 2026-11-16T00:50:47.404446+00:00
Records: 10
Keys: ['id', 'doi', 'title', 'publication_year', 'publication_date', 'authorships', 'cited_by_count', 'open_access', 'abstract_inverted_index', 'primary_location', 'type']
Title: Principles of Artificial Intelligence
Year: 1982
Abstract index present: False
\n S02 query: machine learning
Retrieved: 2026-11-16T00:50:48.676018+00:00
Records: 10
Keys: ['id', 'doi', 'title', 'publication_year', 'publication_date', 'authorships', 'cited_by_count', 'open_access', 'abstract_inverted_index', 'primary_location', 'type']
Title: Machine Learning
Year: 2021
Abstract index present: False


## Part B

In [5]:
# Extract fields and reconstruct available abstracts

def reconstruct_abstract(index):
    if not isinstance(index, dict) or not index:
        return None
    words = {}
    for word, positions in index.items():
        for position in positions:
            words[position] = word
    if not words:
        return None
    return " ".join(words[i] for i in sorted(words))

def prepare_source(path):
    data = json.loads(path.read_text(encoding="utf-8"))
    output, excluded = [], 0
    for item in data["results"]:
        if not isinstance(item, dict) or not item.get("id"):
            excluded += 1
            continue
        authors = [
            (a.get("author") or {}).get("display_name")
            for a in (item.get("authorships") or [])
            if (a.get("author") or {}).get("display_name")
        ]
        location = item.get("primary_location") or {}
        source = location.get("source") or {}
        abstract = reconstruct_abstract(item.get("abstract_inverted_index"))
        output.append({
            "id": item.get("id"), "doi": item.get("doi"),
            "title": item.get("title"), "authors": authors,
            "publication_year": item.get("publication_year"),
            "publication_date": item.get("publication_date"),
            "citation_count": item.get("cited_by_count"),
            "open_access": (item.get("open_access") or {}).get("is_oa"),
            "source_name": source.get("display_name"),
            "abstract": abstract,
            "abstract_available": bool(abstract),
            "provider": "OpenAlex", "source_file": path.name
        })
    return output, excluded

prepared, comparison = [], []
for sid, path in source_paths.items():
    raw = json.loads(path.read_text(encoding="utf-8"))["results"]
    rows, excluded = prepare_source(path)
    prepared.extend(rows)
    comparison.append({
        "source": sid, "raw_records": len(raw),
        "prepared_records": len(rows), "excluded": excluded,
        "abstracts_available": sum(r["abstract_available"] for r in rows),
        "abstracts_missing": sum(not r["abstract_available"] for r in rows)
    })
print("Observed preparation comparison:")
for row in comparison:
    print(row)
print("\\nSample prepared record:")
print(json.dumps(prepared[0], ensure_ascii=False, indent=2)[:1500])

Observed preparation comparison:
{'source': 'S01', 'raw_records': 10, 'prepared_records': 10, 'excluded': 0, 'abstracts_available': 8, 'abstracts_missing': 2}
{'source': 'S02', 'raw_records': 10, 'prepared_records': 10, 'excluded': 0, 'abstracts_available': 5, 'abstracts_missing': 5}
\nSample prepared record:
{
  "id": "https://openalex.org/W2152475379",
  "doi": "https://doi.org/10.1007/978-3-662-09438-9",
  "title": "Principles of Artificial Intelligence",
  "authors": [
    "Nils J. Nilsson"
  ],
  "publication_year": 1982,
  "publication_date": "1982-01-01",
  "citation_count": 3446,
  "open_access": false,
  "source_name": "Springer eBooks",
  "abstract": null,
  "abstract_available": false,
  "provider": "OpenAlex",
  "source_file": "s01_openalex.json"
}


In [6]:
# Export prepared records as JSONL
prepared_path = PROCESSED_DIR / "prepared_json.jsonl"

with prepared_path.open("w", encoding="utf-8") as f:
    for row in prepared:
        f.write(json.dumps(row, ensure_ascii=False) + "\n")

# Read and validate exported JSONL
exported = []

with prepared_path.open("r", encoding="utf-8") as f:
    for line_number, line in enumerate(f, start=1):
        if line.strip():
            try:
                exported.append(json.loads(line))
            except json.JSONDecodeError as e:
                raise ValueError(
                    f"Invalid JSON on line {line_number}: {e}"
                )

# Validation
assert prepared_path.is_file()
assert prepared_path.stat().st_size > 0
assert exported
assert len(exported) == len(prepared)

REQUIRED_KEYS = [
    "id", "doi", "title", "authors",
    "publication_year", "publication_date",
    "citation_count", "open_access", "source_name",
    "abstract", "abstract_available",
    "provider", "source_file"
]

assert all(
    all(key in record for key in REQUIRED_KEYS)
    for record in exported
)

print("\nPATHWAY D completed successfully.")
print("Required keys:", REQUIRED_KEYS)
print("Exported records:", len(exported))
print("Saved:", prepared_path)
print("Validation: PASS")


PATHWAY D completed successfully.
Required keys: ['id', 'doi', 'title', 'authors', 'publication_year', 'publication_date', 'citation_count', 'open_access', 'source_name', 'abstract', 'abstract_available', 'provider', 'source_file']
Exported records: 20
Saved: /content/week3_project/data/processed/prepared_json.jsonl
Validation: PASS


In [7]:
# Save Part B lineage manifest

def sha256(path):
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()

lineage = []
for sid, source in source_paths.items():
    lineage.append({
        "source_id": sid,
        "source_filename": str(source.relative_to(RAW_DIR)),
        "source_sha256": sha256(source),
        "prepared_output": str(prepared_path.relative_to(BASE_DIR)),
        "prepared_sha256": sha256(prepared_path),
        "processing_note": "JSON fields extracted; abstract reconstructed from inverted index when available; missing abstracts flagged",
        "source_version": "OpenAlex API response captured at retrieval time",
        "access_rights": "OpenAlex API; check current API terms",
        "status": "Prepared; quality review pending"
    })
manifest = {
    "pipeline_version": "week3-v1",
    "generated_at_utc": datetime.now(timezone.utc).isoformat(),
    "records": lineage
}
manifest_path = BASE_DIR / "data/week3_manifest.json"
manifest_path.parent.mkdir(parents=True, exist_ok=True)
manifest_path.write_text(json.dumps(manifest, indent=2), encoding="utf-8")
saved = json.loads(manifest_path.read_text(encoding="utf-8"))
assert len(saved["records"]) == len(source_paths)
for row in saved["records"]:
    assert sha256(RAW_DIR / row["source_filename"]) == row["source_sha256"]
    assert sha256(BASE_DIR / row["prepared_output"]) == row["prepared_sha256"]
print("Part B lineage validation: PASS", manifest_path)

Part B lineage validation: PASS /content/week3_project/data/week3_manifest.json


## Part C

In [8]:
# Execute and register the pipeline

def run_pipeline():
    output_dir = PIPE_DIR
    output_dir.mkdir(parents=True, exist_ok=True)

    run_sources = []
    rows = []

    # Reuse the raw JSON snapshots saved during data ingestion
    for sid, query in QUERIES.items():
        source = source_paths[sid]

        wrapped = json.loads(
            source.read_text(encoding="utf-8")
        )

        if not wrapped.get("results"):
            raise ValueError(f"No API results for {query}")

        valid, excluded = prepare_source(source)

        if not valid:
            raise ValueError(f"No valid paper records for {query}")

        rows.extend(valid)

        run_sources.append({
            "source": str(source),
            "sha256": sha256(source),
            "query": query,
            "input_count": len(wrapped["results"]),
            "excluded_count": excluded
        })

    # Export prepared records as JSONL
    output = output_dir / "prepared_json.jsonl"

    with output.open("w", encoding="utf-8") as f:
        for row in rows:
            f.write(json.dumps(row, ensure_ascii=False) + "\n")

    # Read and validate the JSONL file
    check = []

    with output.open("r", encoding="utf-8") as f:
        for line_number, line in enumerate(f, start=1):
            if line.strip():
                try:
                    check.append(json.loads(line))
                except json.JSONDecodeError as e:
                    raise ValueError(
                        f"Invalid JSON on line {line_number}: {e}"
                    )

    if (
        not check
        or len(check) != len(rows)
        or not all(x.get("id") for x in check)
    ):
        raise ValueError("Prepared JSONL validation failed.")

    # Create pipeline manifest
    pm = {
        "pipeline_version": "researchlens-json-v1",
        "executed_at_utc": datetime.now(timezone.utc).isoformat(),
        "sources": run_sources,
        "output": str(output),
        "output_sha256": sha256(output),
        "output_count": len(check),
        "abstracts_available": sum(
            bool(x["abstract"]) for x in check
        ),
        "abstracts_missing": sum(
            not bool(x["abstract"]) for x in check
        ),
        "source_to_output_mapping": [
            {
                "source": x["source"],
                "output": str(output)
            }
            for x in run_sources
        ],
        "processing": (
            "OpenAlex JSON -> field extraction -> "
            "abstract reconstruction -> JSONL"
        )
    }

    pm_path = output_dir / "pipeline_manifest.json"

    pm_path.write_text(
        json.dumps(pm, indent=2),
        encoding="utf-8"
    )

    return {
        "source_hashes": [
            (x["source"], x["sha256"])
            for x in run_sources
        ],
        "output_hash": sha256(output),
        "count": len(check),
        "mapping": pm["source_to_output_mapping"],
        "manifest": pm_path
    }


first_run = run_pipeline()

print("PIPELINE SUCCESS")
print("Output records:", first_run["count"])
print("Manifest:", first_run["manifest"])

PIPELINE SUCCESS
Output records: 20
Manifest: /content/week3_project/data/pipeline_v1/json/pipeline_manifest.json


In [9]:
# Rerun and compare repeatability

second_run = run_pipeline()

assert first_run["source_hashes"] == second_run["source_hashes"]
assert first_run["output_hash"] == second_run["output_hash"]
assert first_run["count"] == second_run["count"]
assert first_run["mapping"] == second_run["mapping"]

print("REPEATABILITY CHECK: PASS")
print("Source hashes, output hash, record count and mappings match.")
print("Retrieval timestamps are expected to differ.")

REPEATABILITY CHECK: PASS
Source hashes, output hash, record count and mappings match.
Retrieval timestamps are expected to differ.
